## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
DATA_ROOT = Path(os.getenv("DATA_ROOT"))
NIH_DIR = DATA_ROOT / "NIH_Chest_X-Rays"
PROJECT_ROOT = Path.cwd().parent

## 2. Make the Split

In [2]:
from src.data import NIH_DISEASES, assign_splits, load_labels

df = assign_splits(load_labels(NIH_DIR), NIH_DIR)
df["split"].value_counts()

split
train    75982
test     25596
val      10542
Name: count, dtype: int64

## 3. Sizes in Images and Patients

In [3]:
df.groupby("split").agg(
    images=("Image Index", "size"),
    patients=("Patient ID", "nunique"),
)

,images,patients
split,,
test,25596,2797
train,75982,24507
val,10542,3501


## 4. Positive Counts Per Class Per Split

In [4]:
df.groupby("split")[NIH_DISEASES].sum().T

split,test,train,val
Atelectasis,3279,7301,979
Cardiomegaly,1069,1491,216
Consolidation,1815,2543,309
Edema,925,1250,128
Effusion,4658,7722,937
Emphysema,1093,1249,174
Fibrosis,435,1098,153
Hernia,86,119,22
Infiltration,6112,12121,1661
Mass,1748,3620,414


## 5. Prevalence Per Split

In [5]:
df.groupby("split")[NIH_DISEASES].mean().T.round(4)

split,test,train,val
Atelectasis,0.1281,0.0961,0.0929
Cardiomegaly,0.0418,0.0196,0.0205
Consolidation,0.0709,0.0335,0.0293
Edema,0.0361,0.0165,0.0121
Effusion,0.1820,0.1016,0.0889
Emphysema,0.0427,0.0164,0.0165
Fibrosis,0.0170,0.0145,0.0145
Hernia,0.0034,0.0016,0.0021
Infiltration,0.2388,0.1595,0.1576
Mass,0.0683,0.0476,0.0393


## 6. AP Fraction and No-Finding Rate Per Split

In [6]:
summary = df.groupby("split").agg(
    ap_frac=("View Position", lambda s: (s == "AP").mean()),
    no_finding=("Finding Labels", lambda s: (s == "No Finding").mean()),
)
summary.round(3)

,ap_frac,no_finding
split,,
test,0.566,0.385
train,0.354,0.581
val,0.324,0.600


## 7. Save the Split

In [7]:
out = PROJECT_ROOT / "splits" / "nih_split.csv"
out.parent.mkdir(exist_ok=True)
df[["Image Index", "Patient ID", "split"]].to_csv(out, index=False)
out

WindowsPath('c:/Users/profb/PROJECTS/cxr-pipeline-v2/splits/nih_split.csv')

## Split decisions
- Test: official NIH `test_list.txt` (patient-disjoint; skewed toward high-follow-up, AP-heavy, sicker patients; used for literature comparability).
- Val: 12.5% of train_val patients, uniform random by patient, seed 42, chosen before any training.
- Hernia val count is seed-sensitive (6–36 across seeds); Hernia val AUROC will not drive model selection.
- Split saved to `splits/nih_split.csv` and committed; all later stages read this file.
- Pneumothorax is ~3.4× more prevalent in test (10.4%) than train (3.1%), likely from post-drain follow-up scans; check drain shortcut with NEATX during error analysis.